# RNN

Coger el lab 4 y adaptar el test para probar una bidireccional

Hay que hacerle unos cambios al test:
1. El h_o ha de ser mas grande

[1,64,hiddendim] --> [2,64,hiddendim]

Bidireccional=true

[batch,sequence dimension, 2<hidden]

Los primeros de izquierda a derecha y los segundos de derecha a izquierda

x. si en algunmomento se igualan los pesos o lo que sea, habra que cambiar el nombre de los weights en cada direccion.

Cambiar las dimensiones etc. (De la RNN y del test) 

Practicar multiple head ..?
The origin of transformers

In [1]:
import torch
import torch.nn as nn
import math

### PRACTICE PROBLEM 1: Simple RNN Cell (Vanilla RNN)

https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html

In [10]:
# ============================================================================
# PRACTICE PROBLEM 1: Simple RNN Cell (Vanilla RNN)
# ============================================================================

class RNNCell(nn.Module):
    """
    Implements a single RNN cell (one time step).
    
    Formula: h_t = tanh(W_ih @ x_t + b_ih + W_hh @ h_{t-1} + b_hh)
    
    This is the building block for RNN layers.
    """
    
    input_size: int
    hidden_size: int
    
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True) -> None:
        """
        Args:
            input_size: Size of input features.
            hidden_size: Size of hidden state.
            bias: If True, adds bias terms.
        
        Returns:
            None.
        """
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Weight for input to hidden
        self.W_ih = nn.Parameter(torch.randn(hidden_size, input_size))
        # Weight for hidden to hidden
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size))
        
        if bias:
            self.b_ih = nn.Parameter(torch.zeros(hidden_size))
            self.b_hh = nn.Parameter(torch.zeros(hidden_size))
        else:
            self.b_ih = None
            self.b_hh = None
        
        self._reset_parameters()
        return None
    
    def _reset_parameters(self):
        """Initialize parameters with Xavier uniform."""
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for weight in self.parameters():
            weight.data.uniform_(-stdv, stdv)
    
    def forward(self, x: torch.Tensor, h: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x: Input at current time step. Dimensions: [batch, input_size].
            h: Hidden state from previous time step. Dimensions: [batch, hidden_size].
               If None, initialize with zeros.
        
        Returns:
            New hidden state. Dimensions: [batch, hidden_size].
        
        Hint:
        1. If h is None: h = torch.zeros(batch_size, hidden_size)
        2. Compute: i2h = x @ W_ih.T + b_ih
        3. Compute: h2h = h @ W_hh.T + b_hh
        4. Combine: h_new = tanh(i2h + h2h)
        5. Return h_new
        """
        # TODO
        batch_size = x.shape[0]
    
        if h is None:
            h =  torch.zeros(batch_size, self.hidden_size)
        
        if self.b_ih is not None:   
            i2h = x @ self.W_ih.T + self.b_ih
        if self.b_hh is not None:
            h2h = h @ self.W_hh.T + self.b_hh
        
        h_new = torch.tanh(i2h + h2h)
        
        return h_new
            


# Test your implementation
def test_rnn_cell():
    batch_size, input_size, hidden_size = 3, 10, 20
    
    x = torch.randn(batch_size, input_size)
    h = torch.randn(batch_size, hidden_size)
    
    my_cell = RNNCell(input_size, hidden_size)
    pytorch_cell = nn.RNNCell(input_size, hidden_size)
    
    # Copy weights
    pytorch_cell.weight_ih.data = my_cell.W_ih.data.clone()
    pytorch_cell.weight_hh.data = my_cell.W_hh.data.clone()
    pytorch_cell.bias_ih.data = my_cell.b_ih.data.clone()
    pytorch_cell.bias_hh.data = my_cell.b_hh.data.clone()
    
    result = my_cell(x, h)
    expected = pytorch_cell(x, h)
    
    print(f"Input shape: {x.shape}")
    print(f"Hidden state shape: {h.shape}")
    print(f"Output shape: {result.shape}")
    print(f"Match: {torch.allclose(result, expected, atol=1e-5)}\n")

test_rnn_cell()


Input shape: torch.Size([3, 10])
Hidden state shape: torch.Size([3, 20])
Output shape: torch.Size([3, 20])
Match: True



#### Learnings

.T - .Transpose 
    
    --> It simply flips rows and columns

(W): (out_features, in_features)

W.T → (in_features, out_features)

###  PRACTICE PROBLEM 2: RNN Layer (1 loop version - over time steps)

In [17]:
# ============================================================================
# PRACTICE PROBLEM 2: RNN Layer (1 loop version - over time steps)
# ============================================================================

class RNN_1Loop(nn.Module):
    """
    Implements RNN layer using 1 loop over time steps.
    
    Processes entire sequence, one time step at a time.
    Input shape: (batch, seq_len, input_size)
    Output shape: (batch, seq_len, hidden_size)
    """
    
    input_size: int
    hidden_size: int
    
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True) -> None:
        """
        Args:
            input_size: Size of input features.
            hidden_size: Size of hidden state.
            bias: If True, adds bias terms.
        
        Returns:
            None.
        """
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Use RNNCell for each time step
        self.cell = RNNCell(input_size, hidden_size, bias)
        
        return None
    
    def forward(self, x: torch.Tensor, h0: torch.Tensor = None) -> tuple:
        """
        Args:
            x: Input sequence. Dimensions: [batch, seq_len, input_size].
            h0: Initial hidden state. Dimensions: [batch, hidden_size].
                If None, initialize with zeros.
        
        Returns:
            Tuple of (output, h_n):
            - output: All hidden states. Dimensions: [batch, seq_len, hidden_size].
            - h_n: Final hidden state. Dimensions: [batch, hidden_size].
        
        Hint (1 loop over time steps):
        1. Extract: batch_size, seq_len, _ = x.shape
        2. Initialize h: if h0 is None, create zeros
        3. Create output list: outputs = []
        4. Loop over t in range(seq_len):
           - Get input at time t: x_t = x[:, t, :]  # Shape: (batch, input_size)
           - Update hidden: h = self.cell(x_t, h)
           - Append to outputs: outputs.append(h.unsqueeze(1))
        5. Stack outputs: output = torch.cat(outputs, dim=1)
        6. Return (output, h)
        """
        # TODO
        batch_size, seq_len, _ = x.shape
        
        if h0 is None:
            h = torch.zeros(batch_size, self.hidden_size, device=x.device)
        else: 
            h=h0
            
        outputs = []
        for t in range(seq_len):
            x_t = x[:, t, :]
            h = self.cell(x_t, h)
            outputs.append(h.unsqueeze(1))
        
        outputs = torch.cat(outputs, dim=1)
        return outputs, h   
            


# Test your implementation
def test_rnn_1loop():
    batch_size, seq_len, input_size, hidden_size = 2, 5, 10, 20
    
    x = torch.randn(batch_size, seq_len, input_size)
    
    my_rnn = RNN_1Loop(input_size, hidden_size)
    pytorch_rnn = nn.RNN(input_size, hidden_size, batch_first=True)
    
    # Copy weights
    pytorch_rnn.weight_ih_l0.data = my_rnn.cell.W_ih.data.clone()
    pytorch_rnn.weight_hh_l0.data = my_rnn.cell.W_hh.data.clone()
    pytorch_rnn.bias_ih_l0.data = my_rnn.cell.b_ih.data.clone()
    pytorch_rnn.bias_hh_l0.data = my_rnn.cell.b_hh.data.clone()
    
    output, h_n = my_rnn(x)
    expected_output, expected_h = pytorch_rnn(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Final hidden shape: {h_n.shape}")
    print(f"Output match: {torch.allclose(output, expected_output, atol=1e-5)}")
    print(f"Hidden match: {torch.allclose(h_n, expected_h.squeeze(0), atol=1e-5)}\n")

test_rnn_1loop()




Input shape: torch.Size([2, 5, 10])
Output shape: torch.Size([2, 5, 20])
Final hidden shape: torch.Size([2, 20])
Output match: True
Hidden match: True



### PRACTICE PROBLEM 3: RNN Layer (2 loops - batch and time)

In [ ]:
# ============================================================================
# PRACTICE PROBLEM 3: RNN Layer (2 loops - batch and time)
# ============================================================================

class RNN_2Loops(nn.Module):
    """
    Implements RNN layer using 2 loops (batch and time).
    
    This is less efficient but shows the complete structure clearly.
    Loops over both batch samples and time steps.
    """
    
    input_size: int
    hidden_size: int
    
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True) -> None:
        """
        Args:
            input_size: Size of input features.
            hidden_size: Size of hidden state.
            bias: If True, adds bias terms.
        
        Returns:
            None.
        """
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        self.W_ih = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size))
        
        if bias:
            self.b_ih = nn.Parameter(torch.zeros(hidden_size))
            self.b_hh = nn.Parameter(torch.zeros(hidden_size))
        else:
            self.b_ih = None
            self.b_hh = None
        
        self._reset_parameters()
        return None
    
    def _reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for weight in self.parameters():
            weight.data.uniform_(-stdv, stdv)
    
    def forward(self, x: torch.Tensor, h0: torch.Tensor = None) -> tuple:
        """
        Args:
            x: Input sequence. Dimensions: [batch, seq_len, input_size].
            h0: Initial hidden state. Dimensions: [batch, hidden_size].
        
        Returns:
            Tuple of (output, h_n).
        
        Hint (2 loops - batch and time):
        1. Extract: batch_size, seq_len, _ = x.shape
        2. Initialize hidden states for all samples:
           if h0 is None: h = torch.zeros(batch_size, hidden_size)
        3. Create output tensor: outputs = torch.zeros(batch_size, seq_len, hidden_size)
        4. Outer loop over t in range(seq_len):
           - Inner loop over b in range(batch_size):
             - Get x_t for sample b: x[b, t, :]  # Shape: (input_size,)
             - Get h for sample b: h[b, :]  # Shape: (hidden_size,)
             - Compute new h[b]: tanh(W_ih @ x + b_ih + W_hh @ h + b_hh)
             - Store in outputs[b, t, :]
        5. Return (outputs, h)
        
        Note: This is inefficient! Use for understanding only.
        """
        # TODO
        pass


# Test your implementation
def test_rnn_2loops():
    batch_size, seq_len, input_size, hidden_size = 2, 3, 4, 5
    
    x = torch.randn(batch_size, seq_len, input_size)
    
    my_rnn = RNN_2Loops(input_size, hidden_size)
    
    output, h_n = my_rnn(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Final hidden shape: {h_n.shape}")
    print(f"Expected output shape: (batch={batch_size}, seq_len={seq_len}, hidden={hidden_size})")
    print(f"Shape correct: {output.shape == (batch_size, seq_len, hidden_size)}\n")

# test_rnn_2loops()




In [ ]:
# ============================================================================
# PRACTICE PROBLEM 4: LSTM Cell
# ============================================================================

class LSTMCell(nn.Module):
    """
    Implements LSTM cell (Long Short-Term Memory).
    
    Has 4 gates: input, forget, gate, output
    Maintains two states: hidden (h) and cell (c)
    
    Formulas:
    i_t = sigmoid(W_ii @ x_t + b_ii + W_hi @ h_{t-1} + b_hi)  # Input gate
    f_t = sigmoid(W_if @ x_t + b_if + W_hf @ h_{t-1} + b_hf)  # Forget gate
    g_t = tanh(W_ig @ x_t + b_ig + W_hg @ h_{t-1} + b_hg)     # Cell gate
    o_t = sigmoid(W_io @ x_t + b_io + W_ho @ h_{t-1} + b_ho)  # Output gate
    c_t = f_t * c_{t-1} + i_t * g_t                            # New cell state
    h_t = o_t * tanh(c_t)                                      # New hidden state
    """
    
    input_size: int
    hidden_size: int
    
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True) -> None:
        """
        Args:
            input_size: Size of input features.
            hidden_size: Size of hidden state.
            bias: If True, adds bias terms.
        
        Returns:
            None.
        """
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Weights for input to hidden (4 gates)
        self.W_ii = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_if = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_ig = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_io = nn.Parameter(torch.randn(hidden_size, input_size))
        
        # Weights for hidden to hidden (4 gates)
        self.W_hi = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.W_hf = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.W_hg = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.W_ho = nn.Parameter(torch.randn(hidden_size, hidden_size))
        
        if bias:
            self.b_ii = nn.Parameter(torch.zeros(hidden_size))
            self.b_if = nn.Parameter(torch.zeros(hidden_size))
            self.b_ig = nn.Parameter(torch.zeros(hidden_size))
            self.b_io = nn.Parameter(torch.zeros(hidden_size))
            self.b_hi = nn.Parameter(torch.zeros(hidden_size))
            self.b_hf = nn.Parameter(torch.zeros(hidden_size))
            self.b_hg = nn.Parameter(torch.zeros(hidden_size))
            self.b_ho = nn.Parameter(torch.zeros(hidden_size))
        else:
            self.register_parameter('b_ii', None)
            self.register_parameter('b_if', None)
            self.register_parameter('b_ig', None)
            self.register_parameter('b_io', None)
            self.register_parameter('b_hi', None)
            self.register_parameter('b_hf', None)
            self.register_parameter('b_hg', None)
            self.register_parameter('b_ho', None)
        
        self._reset_parameters()
        return None
    
    def _reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for weight in self.parameters():
            weight.data.uniform_(-stdv, stdv)
    
    def forward(self, x: torch.Tensor, state: tuple = None) -> tuple:
        """
        Args:
            x: Input at current time step. Dimensions: [batch, input_size].
            state: Tuple of (h, c) from previous time step.
                   h: [batch, hidden_size]
                   c: [batch, hidden_size]
                   If None, initialize both with zeros.
        
        Returns:
            Tuple of (h_new, c_new).
        
        Hint:
        1. If state is None: h = zeros, c = zeros
        2. Compute 4 gates:
           i = sigmoid(x @ W_ii.T + b_ii + h @ W_hi.T + b_hi)
           f = sigmoid(x @ W_if.T + b_if + h @ W_hf.T + b_hf)
           g = tanh(x @ W_ig.T + b_ig + h @ W_hg.T + b_hg)
           o = sigmoid(x @ W_io.T + b_io + h @ W_ho.T + b_ho)
        3. Update cell: c_new = f * c + i * g
        4. Update hidden: h_new = o * tanh(c_new)
        5. Return (h_new, c_new)
        """
        # TODO
        pass


# Test your implementation
def test_lstm_cell():
    batch_size, input_size, hidden_size = 3, 10, 20
    
    x = torch.randn(batch_size, input_size)
    h = torch.randn(batch_size, hidden_size)
    c = torch.randn(batch_size, hidden_size)
    
    my_cell = LSTMCell(input_size, hidden_size)
    pytorch_cell = nn.LSTMCell(input_size, hidden_size)
    
    # Copy weights (PyTorch concatenates all gates)
    pytorch_cell.weight_ih.data = torch.cat([
        my_cell.W_ii.data, my_cell.W_if.data, 
        my_cell.W_ig.data, my_cell.W_io.data
    ], dim=0)
    pytorch_cell.weight_hh.data = torch.cat([
        my_cell.W_hi.data, my_cell.W_hf.data,
        my_cell.W_hg.data, my_cell.W_ho.data
    ], dim=0)
    pytorch_cell.bias_ih.data = torch.cat([
        my_cell.b_ii.data, my_cell.b_if.data,
        my_cell.b_ig.data, my_cell.b_io.data
    ], dim=0)
    pytorch_cell.bias_hh.data = torch.cat([
        my_cell.b_hi.data, my_cell.b_hf.data,
        my_cell.b_hg.data, my_cell.b_ho.data
    ], dim=0)
    
    h_new, c_new = my_cell(x, (h, c))
    expected_h, expected_c = pytorch_cell(x, (h, c))
    
    print(f"Input shape: {x.shape}")
    print(f"Hidden output shape: {h_new.shape}")
    print(f"Cell output shape: {c_new.shape}")
    print(f"Hidden match: {torch.allclose(h_new, expected_h, atol=1e-5)}")
    print(f"Cell match: {torch.allclose(c_new, expected_c, atol=1e-5)}\n")

# test_lstm_cell()


# ============================================================================
# PRACTICE PROBLEM 5: LSTM Layer (1 loop)
# ============================================================================

class LSTM_1Loop(nn.Module):
    """
    Implements LSTM layer using 1 loop over time steps.
    """
    
    input_size: int
    hidden_size: int
    
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True) -> None:
        """
        Args:
            input_size: Size of input features.
            hidden_size: Size of hidden state.
            bias: If True, adds bias terms.
        
        Returns:
            None.
        """
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        self.cell = LSTMCell(input_size, hidden_size, bias)
        
        return None
    
    def forward(self, x: torch.Tensor, state0: tuple = None) -> tuple:
        """
        Args:
            x: Input sequence. Dimensions: [batch, seq_len, input_size].
            state0: Tuple of (h0, c0). Each: [batch, hidden_size].
                    If None, initialize both with zeros.
        
        Returns:
            Tuple of (output, (h_n, c_n)):
            - output: All hidden states. [batch, seq_len, hidden_size].
            - h_n: Final hidden state. [batch, hidden_size].
            - c_n: Final cell state. [batch, hidden_size].
        
        Hint (1 loop):
        1. Extract: batch_size, seq_len, _ = x.shape
        2. Initialize (h, c): if state0 is None, create zeros
        3. Create outputs list: outputs = []
        4. Loop over t in range(seq_len):
           - Get x_t = x[:, t, :]
           - Update (h, c) = self.cell(x_t, (h, c))
           - Append h to outputs
        5. Stack outputs: output = torch.stack(outputs, dim=1)
        6. Return (output, (h, c))
        """
        # TODO
        pass


# Test your implementation
def test_lstm_1loop():
    batch_size, seq_len, input_size, hidden_size = 2, 5, 10, 20
    
    x = torch.randn(batch_size, seq_len, input_size)
    
    my_lstm = LSTM_1Loop(input_size, hidden_size)
    pytorch_lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
    
    # Copy weights
    pytorch_lstm.weight_ih_l0.data = torch.cat([
        my_lstm.cell.W_ii.data, my_lstm.cell.W_if.data,
        my_lstm.cell.W_ig.data, my_lstm.cell.W_io.data
    ], dim=0)
    pytorch_lstm.weight_hh_l0.data = torch.cat([
        my_lstm.cell.W_hi.data, my_lstm.cell.W_hf.data,
        my_lstm.cell.W_hg.data, my_lstm.cell.W_ho.data
    ], dim=0)
    pytorch_lstm.bias_ih_l0.data = torch.cat([
        my_lstm.cell.b_ii.data, my_lstm.cell.b_if.data,
        my_lstm.cell.b_ig.data, my_lstm.cell.b_io.data
    ], dim=0)
    pytorch_lstm.bias_hh_l0.data = torch.cat([
        my_lstm.cell.b_hi.data, my_lstm.cell.b_hf.data,
        my_lstm.cell.b_hg.data, my_lstm.cell.b_ho.data
    ], dim=0)
    
    output, (h_n, c_n) = my_lstm(x)
    expected_output, (expected_h, expected_c) = pytorch_lstm(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Final hidden shape: {h_n.shape}")
    print(f"Final cell shape: {c_n.shape}")
    print(f"Output match: {torch.allclose(output, expected_output, atol=1e-5)}")
    print(f"Hidden match: {torch.allclose(h_n, expected_h.squeeze(0), atol=1e-5)}")
    print(f"Cell match: {torch.allclose(c_n, expected_c.squeeze(0), atol=1e-5)}\n")

# test_lstm_1loop()


# ============================================================================
# PRACTICE PROBLEM 6: Bidirectional RNN (CHALLENGING!)
# ============================================================================

class BidirectionalRNN(nn.Module):
    """
    Implements Bidirectional RNN.
    
    Processes sequence in both directions (forward and backward).
    Concatenates outputs: [h_forward; h_backward]
    
    Output has 2 * hidden_size features.
    """
    
    input_size: int
    hidden_size: int
    
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True) -> None:
        """
        Args:
            input_size: Size of input features.
            hidden_size: Size of hidden state (per direction).
            bias: If True, adds bias terms.
        
        Returns:
            None.
        """
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Forward RNN
        self.rnn_forward = RNN_1Loop(input_size, hidden_size, bias)
        
        # Backward RNN
        self.rnn_backward = RNN_1Loop(input_size, hidden_size, bias)
        
        return None
    
    def forward(self, x: torch.Tensor, h0: tuple = None) -> tuple:
        """
        Args:
            x: Input sequence. Dimensions: [batch, seq_len, input_size].
            h0: Tuple of (h_forward, h_backward). Each: [batch, hidden_size].
                If None, initialize both with zeros.
        
        Returns:
            Tuple of (output, (h_forward_n, h_backward_n)):
            - output: Concatenated outputs. [batch, seq_len, 2*hidden_size].
            - h_forward_n: Final forward hidden. [batch, hidden_size].
            - h_backward_n: Final backward hidden. [batch, hidden_size].
        
        Hint:
        1. Process forward direction:
           output_forward, h_forward = self.rnn_forward(x, h_forward_0)
        2. Flip sequence for backward: x_reversed = torch.flip(x, dims=[1])
        3. Process backward:
           output_backward, h_backward = self.rnn_backward(x_reversed, h_backward_0)
        4. Flip backward output back: output_backward = torch.flip(output_backward, dims=[1])
        5. Concatenate outputs: output = torch.cat([output_forward, output_backward], dim=2)
        6. Return (output, (h_forward, h_backward))
        """
        # TODO
        pass


# Test your implementation
def test_bidirectional_rnn():
    batch_size, seq_len, input_size, hidden_size = 2, 5, 10, 20
    
    x = torch.randn(batch_size, seq_len, input_size)
    
    my_birnn = BidirectionalRNN(input_size, hidden_size)
    pytorch_birnn = nn.RNN(input_size, hidden_size, batch_first=True, bidirectional=True)
    
    # Copy weights for forward direction
    pytorch_birnn.weight_ih_l0.data = my_birnn.rnn_forward.cell.W_ih.data.clone()
    pytorch_birnn.weight_hh_l0.data = my_birnn.rnn_forward.cell.W_hh.data.clone()
    pytorch_birnn.bias_ih_l0.data = my_birnn.rnn_forward.cell.b_ih.data.clone()
    pytorch_birnn.bias_hh_l0.data = my_birnn.rnn_forward.cell.b_hh.data.clone()
    
    # Copy weights for backward direction
    pytorch_birnn.weight_ih_l0_reverse.data = my_birnn.rnn_backward.cell.W_ih.data.clone()
    pytorch_birnn.weight_hh_l0_reverse.data = my_birnn.rnn_backward.cell.W_hh.data.clone()
    pytorch_birnn.bias_ih_l0_reverse.data = my_birnn.rnn_backward.cell.b_ih.data.clone()
    pytorch_birnn.bias_hh_l0_reverse.data = my_birnn.rnn_backward.cell.b_hh.data.clone()
    
    output, (h_forward, h_backward) = my_birnn(x)
    expected_output, expected_h = pytorch_birnn(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Expected output features: 2 * hidden_size = {2 * hidden_size}")
    print(f"Forward hidden shape: {h_forward.shape}")
    print(f"Backward hidden shape: {h_backward.shape}")
    print(f"Output match: {torch.allclose(output, expected_output, atol=1e-5)}\n")

# test_bidirectional_rnn()


# ============================================================================
# PRACTICE PROBLEM 7: GRU Cell (ADVANCED!)
# ============================================================================

class GRUCell(nn.Module):
    """
    Implements GRU cell (Gated Recurrent Unit).
    
    Simpler than LSTM (2 gates instead of 4), no separate cell state.
    
    Formulas:
    r_t = sigmoid(W_ir @ x_t + b_ir + W_hr @ h_{t-1} + b_hr)  # Reset gate
    z_t = sigmoid(W_iz @ x_t + b_iz + W_hz @ h_{t-1} + b_hz)  # Update gate
    n_t = tanh(W_in @ x_t + b_in + r_t * (W_hn @ h_{t-1} + b_hn))  # New gate
    h_t = (1 - z_t) * n_t + z_t * h_{t-1}  # New hidden state
    """
    
    input_size: int
    hidden_size: int
    
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True) -> None:
        """
        Args:
            input_size: Size of input features.
            hidden_size: Size of hidden state.
            bias: If True, adds bias terms.
        
        Returns:
            None.
        """
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Weights for input to hidden (3 gates)
        self.W_ir = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_iz = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_in = nn.Parameter(torch.randn(hidden_size, input_size))
        
        # Weights for hidden to hidden (3 gates)
        self.W_hr = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.W_hz = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.W_hn = nn.Parameter(torch.randn(hidden_size, hidden_size))
        
        if bias:
            self.b_ir = nn.Parameter(torch.zeros(hidden_size))
            self.b_iz = nn.Parameter(torch.zeros(hidden_size))
            self.b_in = nn.Parameter(torch.zeros(hidden_

In [ ]:
# ============================================================================
# PRACTICE PROBLEM 7: GRU Cell (ADVANCED!)
# ============================================================================

class GRUCell(nn.Module):
    """
    Implements GRU cell (Gated Recurrent Unit).
    
    Simpler than LSTM (2 gates instead of 4), no separate cell state.
    
    Formulas:
    r_t = sigmoid(W_ir @ x_t + b_ir + W_hr @ h_{t-1} + b_hr)  # Reset gate
    z_t = sigmoid(W_iz @ x_t + b_iz + W_hz @ h_{t-1} + b_hz)  # Update gate
    n_t = tanh(W_in @ x_t + b_in + r_t * (W_hn @ h_{t-1} + b_hn))  # New gate
    h_t = (1 - z_t) * n_t + z_t * h_{t-1}  # New hidden state
    """
    
    input_size: int
    hidden_size: int
    
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True) -> None:
        """
        Args:
            input_size: Size of input features.
            hidden_size: Size of hidden state.
            bias: If True, adds bias terms.
        
        Returns:
            None.
        """
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Weights for input to hidden (3 gates)
        self.W_ir = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_iz = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_in = nn.Parameter(torch.randn(hidden_size, input_size))
        
        # Weights for hidden to hidden (3 gates)
        self.W_hr = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.W_hz = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.W_hn = nn.Parameter(torch.randn(hidden_size, hidden_size))
        
        if bias:
            self.b_ir = nn.Parameter(torch.zeros(hidden_size))
            self.b_iz = nn.Parameter(torch.zeros(hidden_size))
            self.b_in = nn.Parameter(torch.zeros(hidden_size))
            self.b_hr = nn.Parameter(torch.zeros(hidden_size))
            self.b_hz = nn.Parameter(torch.zeros(hidden_size))
            self.b_hn = nn.Parameter(torch.zeros(hidden_size))
        else:
            self.register_parameter('b_ir', None)
            self.register_parameter('b_iz', None)
            self.register_parameter('b_in', None)
            self.register_parameter('b_hr', None)
            self.register_parameter('b_hz', None)
            self.register_parameter('b_hn', None)
        
        self._reset_parameters()
        return None
    
    def _reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for weight in self.parameters():
            weight.data.uniform_(-stdv, stdv)
    
    def forward(self, x: torch.Tensor, h: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x: Input at current time step. Dimensions: [batch, input_size].
            h: Hidden state from previous time step. Dimensions: [batch, hidden_size].
        
        Returns:
            New hidden state. Dimensions: [batch, hidden_size].
        
        Hint:
        1. If h is None: h = zeros
        2. Compute reset gate:
           r = sigmoid(x @ W_ir.T + b_ir + h @ W_hr.T + b_hr)
        3. Compute update gate:
           z = sigmoid(x @ W_iz.T + b_iz + h @ W_hz.T + b_hz)
        4. Compute new gate:
           n = tanh(x @ W_in.T + b_in + r * (h @ W_hn.T + b_hn))
        5. Compute new hidden:
           h_new = (1 - z) * n + z * h
        6. Return h_new
        """
        # TODO
        pass


# Test your implementation
def test_gru_cell():
    batch_size, input_size, hidden_size = 3, 10, 20
    
    x = torch.randn(batch_size, input_size)
    h = torch.randn(batch_size, hidden_size)
    
    my_cell = GRUCell(input_size, hidden_size)
    pytorch_cell = nn.GRUCell(input_size, hidden_size)
    
    # Copy weights (PyTorch concatenates gates)
    pytorch_cell.weight_ih.data = torch.cat([
        my_cell.W_ir.data, my_cell.W_iz.data, my_cell.W_in.data
    ], dim=0)
    pytorch_cell.weight_hh.data = torch.cat([
        my_cell.W_hr.data, my_cell.W_hz.data, my_cell.W_hn.data
    ], dim=0)
    pytorch_cell.bias_ih.data = torch.cat([
        my_cell.b_ir.data, my_cell.b_iz.data, my_cell.b_in.data
    ], dim=0)
    pytorch_cell.bias_hh.data = torch.cat([
        my_cell.b_hr.data, my_cell.b_hz.data, my_cell.b_hn.data
    ], dim=0)
    
    result = my_cell(x, h)
    expected = pytorch_cell(x, h)
    
    print(f"Input shape: {x.shape}")
    print(f"Hidden state shape: {h.shape}")
    print(f"Output shape: {result.shape}")
    print(f"Match: {torch.allclose(result, expected, atol=1e-5)}\n")

# test_gru_cell()


# ============================================================================
# PRACTICE PROBLEM 8: GRU Layer (1 loop)
# ============================================================================

class GRU_1Loop(nn.Module):
    """
    Implements GRU layer using 1 loop over time steps.
    """
    
    input_size: int
    hidden_size: int
    
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True) -> None:
        """
        Args:
            input_size: Size of input features.
            hidden_size: Size of hidden state.
            bias: If True, adds bias terms.
        
        Returns:
            None.
        """
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        self.cell = GRUCell(input_size, hidden_size, bias)
        
        return None
    
    def forward(self, x: torch.Tensor, h0: torch.Tensor = None) -> tuple:
        """
        Args:
            x: Input sequence. Dimensions: [batch, seq_len, input_size].
            h0: Initial hidden state. Dimensions: [batch, hidden_size].
                If None, initialize with zeros.
        
        Returns:
            Tuple of (output, h_n):
            - output: All hidden states. [batch, seq_len, hidden_size].
            - h_n: Final hidden state. [batch, hidden_size].
        
        Hint (1 loop):
        1. Extract: batch_size, seq_len, _ = x.shape
        2. Initialize h: if h0 is None, create zeros
        3. Create outputs list: outputs = []
        4. Loop over t in range(seq_len):
           - Get x_t = x[:, t, :]
           - Update h = self.cell(x_t, h)
           - Append h to outputs
        5. Stack outputs: output = torch.stack(outputs, dim=1)
        6. Return (output, h)
        """
        # TODO
        pass


# Test your implementation
def test_gru_1loop():
    batch_size, seq_len, input_size, hidden_size = 2, 5, 10, 20
    
    x = torch.randn(batch_size, seq_len, input_size)
    
    my_gru = GRU_1Loop(input_size, hidden_size)
    pytorch_gru = nn.GRU(input_size, hidden_size, batch_first=True)
    
    # Copy weights
    pytorch_gru.weight_ih_l0.data = torch.cat([
        my_gru.cell.W_ir.data, my_gru.cell.W_iz.data, my_gru.cell.W_in.data
    ], dim=0)
    pytorch_gru.weight_hh_l0.data = torch.cat([
        my_gru.cell.W_hr.data, my_gru.cell.W_hz.data, my_gru.cell.W_hn.data
    ], dim=0)
    pytorch_gru.bias_ih_l0.data = torch.cat([
        my_gru.cell.b_ir.data, my_gru.cell.b_iz.data, my_gru.cell.b_in.data
    ], dim=0)
    pytorch_gru.bias_hh_l0.data = torch.cat([
        my_gru.cell.b_hr.data, my_gru.cell.b_hz.data, my_gru.cell.b_hn.data
    ], dim=0)
    
    output, h_n = my_gru(x)
    expected_output, expected_h = pytorch_gru(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Final hidden shape: {h_n.shape}")
    print(f"Output match: {torch.allclose(output, expected_output, atol=1e-5)}")
    print(f"Hidden match: {torch.allclose(h_n, expected_h.squeeze(0), atol=1e-5)}\n")

# test_gru_1loop()




In [ ]:
# ============================================================================
# BONUS: Understanding RNN Dimensions
# ============================================================================

def understand_rnn_dimensions():
    """
    Visual guide to RNN dimensions and data flow.
    """
    print("="*70)
    print("UNDERSTANDING RNN DIMENSIONS")
    print("="*70)
    print("""
Input Shape: (batch, seq_len, input_size)
- batch: Number of sequences in the batch
- seq_len: Length of each sequence (number of time steps)
- input_size: Features per time step

Example: Sentiment analysis with word embeddings
- batch=32: Processing 32 sentences at once
- seq_len=50: Each sentence has 50 words
- input_size=300: Each word is a 300-dim embedding
→ Input shape: (32, 50, 300)

Hidden State: (batch, hidden_size)
- One hidden vector per sample in the batch
- Carries information across time steps

Example:
- hidden_size=128: 128-dimensional hidden representation
→ Hidden shape: (32, 128)

Output Shape: (batch, seq_len, hidden_size)
- Hidden state at EACH time step
- All time steps stacked together

Example:
→ Output shape: (32, 50, 128)

Bidirectional Output: (batch, seq_len, 2*hidden_size)
- Forward and backward hidden states concatenated
→ Bidirectional output: (32, 50, 256)

LSTM Cell State: (batch, hidden_size)
- Additional memory in LSTM
- Same shape as hidden state
→ Cell shape: (32, 128)

Visual Flow (single time step):
    x_t              h_{t-1}
    (batch, input)   (batch, hidden)
         ↓              ↓
         └──→  RNN Cell  ←──┘
                 ↓
               h_t
            (batch, hidden)

Visual Flow (full sequence):
    Input: (batch, seq_len, input)
           ↓
    [Process each time step]
           ↓
    Output: (batch, seq_len, hidden)
    
    Final h_n: (batch, hidden)  ← Just the last time step

Key Points:
- RNN: Only hidden state (h)
- LSTM: Hidden state (h) AND cell state (c)
- GRU: Only hidden state (h), like RNN but with gates
- Bidirectional: Output has 2x hidden_size (concat forward + backward)
    """)

# understand_rnn_dimensions()




In [ ]:
# ============================================================================
# BONUS: Comparison of RNN Variants
# ============================================================================

def compare_rnn_variants():
    """
    Compare different RNN architectures.
    """
    print("="*70)
    print("RNN VARIANTS COMPARISON")
    print("="*70)
    print("""
1. Vanilla RNN:
   - Formula: h_t = tanh(W_ih @ x_t + W_hh @ h_{t-1} + b)
   - States: h (hidden only)
   - Gates: None
   - Parameters: 2 weight matrices + biases
   - Pros: Simple, fast
   - Cons: Vanishing gradient problem, poor for long sequences
   - Use when: Short sequences, simple patterns

2. LSTM (Long Short-Term Memory):
   - Formulas:
     i_t = sigmoid(...)  # Input gate
     f_t = sigmoid(...)  # Forget gate
     g_t = tanh(...)     # Cell gate
     o_t = sigmoid(...)  # Output gate
     c_t = f_t * c_{t-1} + i_t * g_t
     h_t = o_t * tanh(c_t)
   - States: h (hidden) AND c (cell)
   - Gates: 4 (input, forget, cell, output)
   - Parameters: 8 weight matrices + biases
   - Pros: Handles long sequences, solves vanishing gradient
   - Cons: More complex, slower, more parameters
   - Use when: Long sequences, complex dependencies

3. GRU (Gated Recurrent Unit):
   - Formulas:
     r_t = sigmoid(...)  # Reset gate
     z_t = sigmoid(...)  # Update gate
     n_t = tanh(...)     # New gate
     h_t = (1 - z_t) * n_t + z_t * h_{t-1}
   - States: h (hidden only)
   - Gates: 3 (reset, update, new)
   - Parameters: 6 weight matrices + biases
   - Pros: Similar performance to LSTM, fewer parameters, faster
   - Cons: Less flexible than LSTM
   - Use when: Good balance of performance and efficiency

Parameter Count Comparison (input_size=100, hidden_size=128):
- RNN:  2 × (128 × 100 + 128 × 128) = 58,368 parameters
- LSTM: 4 × (128 × 100 + 128 × 128) = 116,736 parameters
- GRU:  3 × (128 × 100 + 128 × 128) = 87,552 parameters

Speed Comparison (approximate, relative to RNN):
- RNN:  1.0x (baseline)
- GRU:  1.5x slower
- LSTM: 2.0x slower

Performance on Long Sequences:
- RNN:  Poor (vanishing gradient)
- GRU:  Good
- LSTM: Good (slightly better than GRU on very long sequences)

Common Practice:
- Start with GRU (good balance)
- Use LSTM if you need maximum performance on long sequences
- Use RNN only for very short sequences or when speed is critical
    """)

# compare_rnn_variants()




In [ ]:
# ============================================================================
# BONUS: Run all tests
# ============================================================================

def run_all_tests():
    """Run all test functions"""
    print("="*70)
    print("RUNNING ALL RNN TESTS")
    print("="*70 + "\n")
    
    tests = [
        ("RNN Cell", test_rnn_cell),
        ("RNN (1 loop)", test_rnn_1loop),
        ("RNN (2 loops)", test_rnn_2loops),
        ("LSTM Cell", test_lstm_cell),
        ("LSTM (1 loop)", test_lstm_1loop),
        ("Bidirectional RNN", test_bidirectional_rnn),
        ("GRU Cell", test_gru_cell),
        ("GRU (1 loop)", test_gru_1loop),
    ]
    
    for name, test_func in tests:
        print(f"--- {name} ---")
        try:
            test_func()
        except Exception as e:
            print(f"❌ Error: {e}\n")
    
    print("="*70)
    print("TESTS COMPLETE")
    print("="*70)

# Uncomment to run all tests:
# run_all_tests()
# understand_rnn_dimensions()
# compare_rnn_variants()


In [ ]:
# ============================================================================
# SOLUTIONS (Don't look until you've tried!)
# ============================================================================

# Scroll down for solutions...
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#

"""
SOLUTIONS:

# 1. RNNCell
def forward(self, x, h=None):
    batch_size = x.shape[0]
    if h is None:
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)
    
    # Compute transformations
    i2h = x @ self.W_ih.T
    h2h = h @ self.W_hh.T
    
    # Add biases
    if self.b_ih is not None:
        i2h = i2h + self.b_ih
    if self.b_hh is not None:
        h2h = h2h + self.b_hh
    
    # Combine and activate
    h_new = torch.tanh(i2h + h2h)
    return h_new


# 2. RNN_1Loop
def forward(self, x, h0=None):
    batch_size, seq_len, _ = x.shape
    
    if h0 is None:
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)
    else:
        h = h0
    
    outputs = []
    for t in range(seq_len):
        x_t = x[:, t, :]
        h = self.cell(x_t, h)
        outputs.append(h.unsqueeze(1))
    
    output = torch.cat(outputs, dim=1)
    return output, h


# 3. RNN_2Loops
def forward(self, x, h0=None):
    batch_size, seq_len, _ = x.shape
    
    if h0 is None:
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)
    else:
        h = h0
    
    outputs = torch.zeros(batch_size, seq_len, self.hidden_size, device=x.device)
    
    for t in range(seq_len):
        for b in range(batch_size):
            x_t = x[b, t, :]
            h_prev = h[b, :]
            
            i2h = x_t @ self.W_ih.T
            h2h = h_prev @ self.W_hh.T
            
            if self.b_ih is not None:
                i2h = i2h + self.b_ih
            if self.b_hh is not None:
                h2h = h2h + self.b_hh
            
            h[b, :] = torch.tanh(i2h + h2h)
            outputs[b, t, :] = h[b, :]
    
    return outputs, h


# 4. LSTMCell
def forward(self, x, state=None):
    batch_size = x.shape[0]
    
    if state is None:
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)
        c = torch.zeros(batch_size, self.hidden_size, device=x.device)
    else:
        h, c = state
    
    # Input gate
    i = torch.sigmoid(x @ self.W_ii.T + self.b_ii + h @ self.W_hi.T + self.b_hi)
    
    # Forget gate
    f = torch.sigmoid(x @ self.W_if.T + self.b_if + h @ self.W_hf.T + self.b_hf)
    
    # Cell gate
    g = torch.tanh(x @ self.W_ig.T + self.b_ig + h @ self.W_hg.T + self.b_hg)
    
    # Output gate
    o = torch.sigmoid(x @ self.W_io.T + self.b_io + h @ self.W_ho.T + self.b_ho)
    
    # New cell state
    c_new = f * c + i * g
    
    # New hidden state
    h_new = o * torch.tanh(c_new)
    
    return h_new, c_new


# 5. LSTM_1Loop
def forward(self, x, state0=None):
    batch_size, seq_len, _ = x.shape
    
    if state0 is None:
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)
        c = torch.zeros(batch_size, self.hidden_size, device=x.device)
    else:
        h, c = state0
    
    outputs = []
    for t in range(seq_len):
        x_t = x[:, t, :]
        h, c = self.cell(x_t, (h, c))
        outputs.append(h.unsqueeze(1))
    
    output = torch.cat(outputs, dim=1)
    return output, (h, c)


# 6. BidirectionalRNN
def forward(self, x, h0=None):
    if h0 is None:
        h_forward = None
        h_backward = None
    else:
        h_forward, h_backward = h0
    
    # Forward direction
    output_forward, h_forward_n = self.rnn_forward(x, h_forward)
    
    # Backward direction (flip sequence)
    x_reversed = torch.flip(x, dims=[1])
    output_backward, h_backward_n = self.rnn_backward(x_reversed, h_backward)
    
    # Flip backward output back to original order
    output_backward = torch.flip(output_backward, dims=[1])
    
    # Concatenate outputs
    output = torch.cat([output_forward, output_backward], dim=2)
    
    return output, (h_forward_n, h_backward_n)


# 7. GRUCell
def forward(self, x, h=None):
    batch_size = x.shape[0]
    if h is None:
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)
    
    # Reset gate
    r = torch.sigmoid(x @ self.W_ir.T + self.b_ir + h @ self.W_hr.T + self.b_hr)
    
    # Update gate
    z = torch.sigmoid(x @ self.W_iz.T + self.b_iz + h @ self.W_hz.T + self.b_hz)
    
    # New gate
    n = torch.tanh(x @ self.W_in.T + self.b_in + r * (h @ self.W_hn.T + self.b_hn))
    
    # New hidden state
    h_new = (1 - z) * n + z * h
    
    return h_new


# 8. GRU_1Loop
def forward(self, x, h0=None):
    batch_size, seq_len, _ = x.shape
    
    if h0 is None:
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)
    else:
        h = h0
    
    outputs = []
    for t in range(seq_len):
        x_t = x[:, t, :]
        h = self.cell(x_t, h)
        outputs.append(h.unsqueeze(1))
    
    output = torch.cat(outputs, dim=1)
    return output, h
"""